# [LELEC2870] Machine Learning Project — Heart failure in the Smurf society
# Part 2 — Nonlinear Models

---

This notebook uses Part 1 as a baseline to beat.

**Constraints** (from the project guidelines):
- at most 6 features from the original tabular dataset;
- evaluation criterion = RW-RMSE, with weights `w = 1 + y/0.1`;
- `X_test.csv` / `y_test.csv` are the test set: used once, at the very end

In [46]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer

from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression

from sklearn.feature_selection import mutual_info_regression, SequentialFeatureSelector
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Setup OK")

Setup OK


## 1. Data loading

- We reload the raw CSV files

- Before any modelling we verify a set of explicit assumptions just to check if every data has its correspondance label around the columns. Each assumption is written as an assert, so the notebook stops immediately if a later edit breaks one, instead of silently producing a wrong number.

In [47]:
DATA = "data_student/data_labeled/"

X_train_raw = pd.read_csv(DATA + "X_train.csv")
X_test_raw  = pd.read_csv(DATA + "X_test.csv")
y_train = pd.read_csv(DATA + "y_train.csv", header=None).values.ravel()
y_test  = pd.read_csv(DATA + "y_test.csv",  header=None).values.ravel()

# `img_filename` points to the heart scan used in Part 3. Part 2 is tabular only.
IMG_COL = "img_filename"
X_train_raw = X_train_raw.drop(columns=[IMG_COL])
X_test_raw  = X_test_raw.drop(columns=[IMG_COL])

# The 13 original tabular features, split by type.
CATEGORICAL = ["profession", "sarsaparilla", "smurfberry_liquor", "smurfin_donuts"]
NUMERIC     = [c for c in X_train_raw.columns if c not in CATEGORICAL]

# The three consumption variables are ordered scales, not unordered labels.
ORDINAL     = ["sarsaparilla", "smurfberry_liquor", "smurfin_donuts"]
NOMINAL     = ["profession"]
ORD_LEVELS  = ["very_low", "low", "moderate", "high", "very_high"]

print(f"X_train {X_train_raw.shape}   X_test {X_test_raw.shape}")
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical = "
      f"{len(NUMERIC)+len(CATEGORICAL)} original features")
print(f"numeric    : {NUMERIC}")
print(f"categorical: {CATEGORICAL}")

X_train (1000, 13)   X_test (500, 13)
9 numeric + 4 categorical = 13 original features
numeric    : ['age', 'blood_pressure', 'calcium', 'cholesterol', 'hemoglobin', 'height', 'potassium', 'vitamin_D', 'weight']
categorical: ['profession', 'sarsaparilla', 'smurfberry_liquor', 'smurfin_donuts']


## 1.2 Different classes of nonlinear models

- **Distance-based models**: these build their prediction out of the quantity

$$\|\mathbf{x} - \mathbf{x}_i\| \;=\; \sqrt{\sum_{j=1}^{p}\left(x_j - x_{ij}\right)^2}$$

  which is the Euclidean distance between a new Smurf $\mathbf{x}$ and a training
  Smurf $\mathbf{x}_i$. **Members: k-NN, SVR, MLP.**

- **Split-based models**: these methods ask a yes/no question about one feature at a
  time and get back a zero or a one,

$$\mathbb{1}\left[x_j \le t\right] \;=\;
\begin{cases} 1 & \text{if } x_j \le t \\[2pt] 0 & \text{otherwise}\end{cases}$$

  **Members: decision tree, random forest, gradient boosting.**

### Why does this division into classes matter?

- This difference matters because it is possible to see that the **distance-based
  models** improve their score when the data are standardised, and in the case of the
  second group nothing increases or decreases.

- This happens with the distance-based models because of the sum inside the root,
  which can increase or decrease the weight of the features in a wrong way: a feature
  measured in large units contributes large squared terms and dominates the distance.
  The standardisation makes all the features valued in the same measure, the
  **standard deviation**, so that every feature contributes on an equal footing.

- However, on the other hand, a tree is always **unit-invariant**: rescaling a feature
  as $x_j \mapsto a x_j + c$ moves the threshold by the same amount, so the same
  Smurfs fall on the same side of the split and the partition is identical.

### Extrapolation

- This topic is important in this part of the project since in Part 1 a linear model
  just follows the line that the model projected, even far away from the data.
  However, a nonlinear model will behave in a different way when it crosses the edges:
  tree-based models and k-NN become flat and repeat the value they saw at the edge,
  while SVR and the MLP keep moving in a direction that nothing in the data constrains.

- So, it is important to find the data outside the min and max in the test Smurfs
  (the min and max are computed on the training data set, so by definition every
  training Smurf is inside and only the test Smurfs can fall outside).

In [48]:
rows = []
for c in NUMERIC:
    lo, hi = X_train_raw[c].min(), X_train_raw[c].max()
    n_out = int(((X_test_raw[c] < lo) | (X_test_raw[c] > hi)).sum())
    rows.append({"feature": c,
                 "train_min": round(lo, 2),  "train_max": round(hi, 2),
                 "test_min": round(X_test_raw[c].min(), 2),
                 "test_max": round(X_test_raw[c].max(), 2),
                 "test_outside": n_out})

extrap = pd.DataFrame(rows).set_index("feature")
outside_any = np.any([((X_test_raw[c] < X_train_raw[c].min()) |
                       (X_test_raw[c] > X_train_raw[c].max())).values
                      for c in NUMERIC], axis=0)

print(extrap.to_string())
print(f"\nTest Smurfs outside the training range on >=1 feature: "
      f"{outside_any.sum()} / {len(X_test_raw)} ({100*outside_any.mean():.1f}%)")

                train_min  train_max  test_min  test_max  test_outside
feature                                                               
age                 26.00     180.00     32.00    180.00             0
blood_pressure      88.03     163.62     84.37    154.44             1
calcium              1.78       3.11      1.88      3.02             0
cholesterol         39.37     189.06     50.72    188.79             0
hemoglobin           8.02      20.04      7.69     20.39             3
height               6.36       8.93      6.52      8.75             0
potassium            2.78       5.57      2.68      5.89             3
vitamin_D           11.39      48.61     15.27     47.52             0
weight              60.13     151.35     63.95    153.36             2

Test Smurfs outside the training range on >=1 feature: 9 / 500 (1.8%)


| | |
|---|---|
| test Smurfs outside the training range on $\geq 1$ feature | **9 of 500 (1.8%)** |
| largest distance past the edge | **0.66 standard deviations** |
| features concerned | `potassium` (3), `hemoglobin` (3), `weight` (2), `blood_pressure` (1) |

  Since only 1.8% of the test Smurfs are concerned and never by much, extrapolation
  has a negligible effect on our final generalization estimate.

## 2. Preprocessing rebuilt for nonlinear models

- As we saw before, the two classes above will need different ways of dealing with the
  data. On the one hand, for the distance class, the function `make_preprocessor` will
  be called with the argument `scale=True` in order to standardise the data. However,
  on the other hand, the split class will not need that, so the function will keep
  doing what was done in Part 1: it will just modify the columns with categorical
  labels into numbers in a range of 0 to 4, and one-hot encoding for professions.
  **The numerical and ordinal columns are standardised only in the distance case; the
  one-hot columns stay as 0 and 1 in both cases.**

In [49]:
def make_preprocessor(features, scale):

    #features : list of original column names (at most 6, per the guidelines)
    #scale    : True  -> standardise numeric and ordinal columns (distance-based models)
               #False -> leave them in their original units (split-based models)
    f_num = [c for c in features if c in NUMERIC]
    f_ord = [c for c in features if c in ORDINAL]
    f_nom = [c for c in features if c in NOMINAL]

    blocks = []

    if f_num:
        blocks.append(("num", StandardScaler() if scale else "passthrough", f_num))

    if f_ord:
        # The five levels are an ordered scale, so we map them to 0..4 and keep that
        # order. A split-based model uses only the order; a distance-based model also
        # needs the result standardised, like any other numeric column.
        steps = [("encode", OrdinalEncoder(categories=[ORD_LEVELS] * len(f_ord)))]
        if scale:
            steps.append(("scale", StandardScaler()))
        blocks.append(("ord", Pipeline(steps), f_ord))

    if f_nom:
        # `profession` has no natural order, so it is one-hot encoded.
        # drop="first" removes the redundant column (dummy variable trap).
        blocks.append(("nom", OneHotEncoder(drop="first", sparse_output=False), f_nom))

    return ColumnTransformer(blocks, remainder="drop")


# Which models need scaling. Used everywhere below.
NEEDS_SCALING = {"knn": True, "svr": True, "mlp": True,
                 "tree": False, "forest": False, "boosting": False}

print("Counting rule (guidelines footnote): one-hot encoding `profession` produces")
print("5 columns but still counts as 1 of the 6 allowed original features.")

Counting rule (guidelines footnote): one-hot encoding `profession` produces
5 columns but still counts as 1 of the 6 allowed original features.


### 2.1 Evidence for the two-pipeline decision

- Now, we are going to prove that the two families need different treatments of the
  data.
- Below, the function will perform the cross-validation of all the models, doing five
  rounds, each one partitioning the data into 800 training Smurfs + 200 validation
  Smurfs, and taking the average of the five scores.
- Each model is run **twice** — once with `scale=False` and once with `scale=True` —
  and nothing else changes between the two runs, so any difference in the score can
  only have been caused by the standardisation.

In [50]:
def quick_cv(model, scale, features, sample_weight=True):
    #5-fold CV RW-RMSE. Preprocessing is refitted inside every fold
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr, va in kf.split(X_train_raw):
        pipe = Pipeline([("prep", make_preprocessor(features, scale)),
                         ("model", model)])
        if sample_weight:
            pipe.fit(X_train_raw.iloc[tr], y_train[tr],
                     model__sample_weight=1.0 + y_train[tr] / 0.1)
        else:
            pipe.fit(X_train_raw.iloc[tr], y_train[tr])
        pred = pipe.predict(X_train_raw.iloc[va])
        w = 1.0 + y_train[va] / 0.1
        scores.append(np.sqrt(np.sum(w * (y_train[va] - pred) ** 2) / np.sum(w)))
    return float(np.mean(scores))


ALL_FEATURES = NUMERIC + CATEGORICAL

tests = [
    ("k-NN (k=25)",         KNeighborsRegressor(n_neighbors=25),                     False),
    ("SVR (RBF, gamma=.05)", SVR(C=1.0, gamma=0.05),                                 False),
    ("Decision tree (d=5)", DecisionTreeRegressor(max_depth=5, random_state=0),      True),
    ("Random forest",       RandomForestRegressor(n_estimators=300, min_samples_leaf=10,
                                                  random_state=0, n_jobs=1),         True),
    ("Gradient boosting",   HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                                          max_leaf_nodes=15, min_samples_leaf=30,
                                                          random_state=0),           True),
]

print(f"{'model':24s} {'raw units':>11s} {'standardised':>14s} {'gain':>9s}")
print("-" * 62)
for name, mdl, use_w in tests:
    raw = quick_cv(mdl, scale=False, features=ALL_FEATURES, sample_weight=use_w)
    std = quick_cv(mdl, scale=True,  features=ALL_FEATURES, sample_weight=use_w)
    print(f"{name:24s} {raw:11.5f} {std:14.5f} {100*(raw-std)/raw:+8.1f}%")

model                      raw units   standardised      gain
--------------------------------------------------------------
k-NN (k=25)                  0.07351        0.06948     +5.5%
SVR (RBF, gamma=.05)         0.09904        0.06114    +38.3%
Decision tree (d=5)          0.06747        0.06747     -0.0%
Random forest                0.05719        0.05718     +0.0%
Gradient boosting            0.05290        0.05289     +0.0%


## 3. The evaluation criterion: RW-RMSE

The project is scored with the risk-weighted RMSE

$$\text{RW-RMSE}(y,\hat y)=\sqrt{\frac{\sum_i w_i (y_i-\hat y_i)^2}{\sum_i w_i}},
\qquad w_i = 1 + \frac{y_i}{0.1}$$

The weights depend only on the **true** risk, never on the prediction. That is what
makes them legitimate to use during *training* and not only during *evaluation*:
at training time the true risk is known.

| a Smurf whose true risk is | gets weight | counts as |
|---|---|---|
| $y = 0.00$ | $1.0$ | 1 Smurf |
| $y = 0.10$ | $2.0$ | 2 Smurfs |
| $y = 0.49$ | $5.9$ | 5.9 Smurfs |

- The goal of this section is to tell the model that sick Smurfs matter more than
  healthy ones. Consequently, that is the idea of `sample_weight`, because it allows
  sick Smurfs to get more weight in the average than healthier ones. In this way, the
  model is able to compute weighted averages instead of plain ones.

- However, some models do not allow `sample_weight`, like k-NN, which just stores the
  data and analyses the Smurfs' neighbours. As you cannot say "count this Smurf more
  than the others", it is possible to replicate them, repeating each Smurf as many
  times as its weight says. A squared-error objective treats a row repeated $k$ times
  exactly like a row carrying weight $k$, so both routes give the same result.

- On our scikit-learn version, the MLP also refuses `sample_weight`, so support is
  checked at runtime rather than assumed, and the two models that refuse it are
  weighted by replication instead.

In [51]:
from sklearn.base import clone

def rw_rmse(y_true, y_pred):
    """Risk-weighted RMSE, the official evaluation criterion of the project."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    w = 1.0 + y_true / 0.1
    return float(np.sqrt(np.sum(w * (y_true - y_pred) ** 2) / np.sum(w)))


def rmse(y_true, y_pred):
    """Unweighted RMSE, reported alongside RW-RMSE for comparison."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def risk_weights(y):
    """The weights the criterion assigns. They depend only on the TRUE risk,
    which is why they can legitimately be used during training as well."""
    return 1.0 + np.asarray(y).ravel() / 0.1


# Scorer object for GridSearchCV / cross_val_score.
# greater_is_better=False because RW-RMSE is an error: sklearn negates it internally,
# so "highest score" still means "lowest error".
rw_rmse_scorer = make_scorer(rw_rmse, greater_is_better=False)

w_train = risk_weights(y_train)
print(f"weights on the training set: min {w_train.min():.2f}  "
      f"max {w_train.max():.2f}  mean {w_train.mean():.2f}")
print(f"a Smurf at y=0.00 counts {1.0:.1f}x,  at y=0.10 counts {1+0.10/0.1:.1f}x,  "
      f"at y=0.49 counts {1+0.49/0.1:.1f}x")

weights on the training set: min 1.00  max 5.90  mean 1.81
a Smurf at y=0.00 counts 1.0x,  at y=0.10 counts 2.0x,  at y=0.49 counts 5.9x


- k-NN does not allow `sample_weight` because there is no equation being minimised,
  so there is nowhere for a weight to go.

In [52]:
import inspect

def supports_sample_weight(estimator):
    """Ask the estimator itself, rather than assuming. Support differs
    between scikit-learn versions, so this is checked at runtime."""
    try:
        return "sample_weight" in inspect.signature(estimator.fit).parameters
    except (TypeError, ValueError):
        return False


def replicate_by_weight(X_df, y_arr, factor=3):
    """Weighting by duplication, for models whose fit() takes no sample_weight.

    A squared-error objective treats a row repeated k times exactly like a row
    carrying weight k, so repeating each Smurf round(w * factor) times
    reproduces the weighted objective using integer multiplicities only.
    """
    reps = np.maximum(1, np.round(risk_weights(y_arr) * factor).astype(int))
    idx = np.repeat(np.arange(len(y_arr)), reps)
    return X_df.iloc[idx], y_arr[idx]


def fit_weighted(pipe, X_df, y_arr):
    """Fit `pipe` so that its objective carries the RW-RMSE weights, using
    sample_weight when the final estimator accepts it and replication when not."""
    name, final = pipe.steps[-1]
    if supports_sample_weight(final):
        pipe.fit(X_df, y_arr, **{f"{name}__sample_weight": risk_weights(y_arr)})
    else:
        Xr, yr = replicate_by_weight(X_df, y_arr)
        pipe.fit(Xr, yr)
    return pipe


CANDIDATES = {
    "k-NN":              KNeighborsRegressor(),
    "Decision tree":     DecisionTreeRegressor(),
    "Random forest":     RandomForestRegressor(),
    "Gradient boosting": HistGradientBoostingRegressor(),
    "SVR":               SVR(),
    "MLP":               MLPRegressor(),
}

print(f"{'model':20s} {'accepts sample_weight?':>24s} {'weighting route':>20s}")
print("-" * 66)
for name, est in CANDIDATES.items():
    ok = supports_sample_weight(est)
    print(f"{name:20s} {('yes' if ok else 'NO'):>24s} "
          f"{('sample_weight' if ok else 'replication'):>20s}")

model                  accepts sample_weight?      weighting route
------------------------------------------------------------------
k-NN                                       NO          replication
Decision tree                             yes        sample_weight
Random forest                             yes        sample_weight
Gradient boosting                         yes        sample_weight
SVR                                       yes        sample_weight
MLP                                        NO          replication


- Above, each model was trained twice: one time without any specific weight, and the
  other with `sample_weight` or replication, and we can see the gain.

**What is the gain?**

$$\text{gain} = \frac{\text{unweighted} - \text{weighted}}{\text{unweighted}}$$

| column | what was done |
|---|---|
| **unweighted** | the model was never told that sick Smurfs matter more |
| **weighted** | the model was told — via `sample_weight` or replication |

Both columns are scored with RW-RMSE, so the exam is the same in both cases and only
the training changed. The gain is therefore the percentage of error that disappeared
purely because the model was told what the metric cares about.

In [53]:
def cv_weighted(model, scale, features, weighted=True):
    """5-fold CV RW-RMSE. `weighted=True` puts the criterion's weights into
    the training objective; `weighted=False` trains unweighted."""
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr, va in kf.split(X_train_raw):
        pipe = Pipeline([("prep", make_preprocessor(features, scale)),
                         ("model", clone(model))])
        if weighted:
            fit_weighted(pipe, X_train_raw.iloc[tr], y_train[tr])
        else:
            pipe.fit(X_train_raw.iloc[tr], y_train[tr])
        scores.append(rw_rmse(y_train[va], pipe.predict(X_train_raw.iloc[va])))
    return float(np.mean(scores))


weighting_tests = [
    ("k-NN (k=25)",         KNeighborsRegressor(n_neighbors=25),                  True),
    ("Decision tree (d=5)", DecisionTreeRegressor(max_depth=5, random_state=0),   False),
    ("Random forest",       RandomForestRegressor(n_estimators=300, min_samples_leaf=10,
                                                  random_state=0, n_jobs=1),      False),
    ("Gradient boosting",   HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                                          max_leaf_nodes=15,
                                                          min_samples_leaf=30,
                                                          random_state=0),        False),
]

print(f"{'model':22s} {'unweighted':>11s} {'weighted':>10s} {'gain':>9s}")
print("-" * 56)
for name, mdl, sc in weighting_tests:
    u = cv_weighted(mdl, sc, ALL_FEATURES, weighted=False)
    w = cv_weighted(mdl, sc, ALL_FEATURES, weighted=True)
    print(f"{name:22s} {u:11.5f} {w:10.5f} {100*(u-w)/u:+8.1f}%")

model                   unweighted   weighted      gain
--------------------------------------------------------
k-NN (k=25)                0.06948    0.06310     +9.2%
Decision tree (d=5)        0.06810    0.06747     +0.9%
Random forest              0.05981    0.05719     +4.4%
Gradient boosting          0.05392    0.05290     +1.9%


## 4. Splitting protocol for model selection

- The 1500 labeled Smurfs carry three distinct roles:

| set | size | role | how often it may be used |
|---|---|---|---|
| **training folds** | 800 per fold | fit model parameters | thousands of times |
| **validation folds** | 200 per fold | choose model and hyperparameters | thousands of times |
| **test set** | 500 | estimate generalization error | **exactly once** |

- This section's goal is to define a fixed CV object, a `cv_score()` function and a
  proof that paired comparisons work, so first of all we separated the training and
  validation sets into 5 groups below.

- In order to do this, it is necessary to choose a model — we chose boosting below,
  but it could be another one — and the idea is to measure with the RW-RMSE how much
  the score can change during this process, depending only on how the folds are drawn.

- The answer is that the same model scores anywhere over a range of **0.00289**
  across 12 different fold partitions, so any difference between two models smaller
  than roughly **0.003 is not interpretable**. However, the *paired* difference is
  stable: boosting beat the decision tree in **12/12** partitions by
  **0.01456 ± 0.00227**. This is why a single `KFold` object is reused for every
  candidate, so that all comparisons are paired and the fold-to-fold noise cancels.

In [54]:
# The single cross-validation object used for EVERY model in this notebook.
# Fixing it means all candidates are scored on exactly the same 5 splits, so the
# comparison between them is paired and the fold-to-fold noise cancels.
CV = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for i, (tr, va) in enumerate(CV.split(X_train_raw), start=1):
    print(f"  fold {i}: train {len(tr):4d} Smurfs   validate {len(va):4d} Smurfs   "
          f"mean risk in validation fold = {y_train[va].mean():.4f}")

  fold 1: train  800 Smurfs   validate  200 Smurfs   mean risk in validation fold = 0.0821
  fold 2: train  800 Smurfs   validate  200 Smurfs   mean risk in validation fold = 0.0837
  fold 3: train  800 Smurfs   validate  200 Smurfs   mean risk in validation fold = 0.0753
  fold 4: train  800 Smurfs   validate  200 Smurfs   mean risk in validation fold = 0.0747
  fold 5: train  800 Smurfs   validate  200 Smurfs   mean risk in validation fold = 0.0875


In [55]:
def cv_score(model, scale, features, return_folds=False):
    """5-fold CV RW-RMSE on the training set, using the fixed CV object.
    Preprocessing and the weighted fit happen inside each fold."""
    folds = []
    for tr, va in CV.split(X_train_raw):
        pipe = Pipeline([("prep", make_preprocessor(features, scale)),
                         ("model", clone(model))])
        fit_weighted(pipe, X_train_raw.iloc[tr], y_train[tr])
        folds.append(rw_rmse(y_train[va], pipe.predict(X_train_raw.iloc[va])))
    folds = np.array(folds)
    if return_folds:
        return folds
    return folds.mean(), folds.std()


m, s = cv_score(HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                              max_leaf_nodes=15, min_samples_leaf=30,
                                              random_state=0),
                scale=False, features=ALL_FEATURES)
print(f"gradient boosting, all 13 features")
print(f"  CV RW-RMSE          {m:.5f}")
print(f"  fold-to-fold sd     {s:.5f}   (how much one fold differs from another)")
print(f"  std. error of mean  {s/np.sqrt(5):.5f}   (uncertainty on the number above)")

gradient boosting, all 13 features
  CV RW-RMSE          0.05290
  fold-to-fold sd     0.00530   (how much one fold differs from another)
  std. error of mean  0.00237   (uncertainty on the number above)


- Here we can see that boosting wins in every try against a simple tree algorithm,
  in all 12 partitions.

In [56]:
# How much does a CV score move for reasons that have nothing to do with the model?
# We re-score ONE fixed model under 12 different fold partitions.
gb = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                   max_leaf_nodes=15, min_samples_leaf=30,
                                   random_state=0)
dt = DecisionTreeRegressor(max_depth=5, random_state=0)

def cv_with_seed(model, scale, features, seed):
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    folds = []
    for tr, va in kf.split(X_train_raw):
        pipe = Pipeline([("prep", make_preprocessor(features, scale)),
                         ("model", clone(model))])
        fit_weighted(pipe, X_train_raw.iloc[tr], y_train[tr])
        folds.append(rw_rmse(y_train[va], pipe.predict(X_train_raw.iloc[va])))
    return float(np.mean(folds))

print(f"{'fold seed':>9s} {'boosting':>10s} {'tree':>10s} {'difference':>12s}")
diffs = []
for seed in range(12):
    a = cv_with_seed(gb, False, ALL_FEATURES, seed)
    b = cv_with_seed(dt, False, ALL_FEATURES, seed)
    diffs.append(b - a)
    print(f"{seed:9d} {a:10.5f} {b:10.5f} {b - a:12.5f}")

diffs = np.array(diffs)
print(f"\nBoosting's own score ranges over {max(...)-min(...) if False else ''}"
      f"{np.ptp([cv_with_seed(gb, False, ALL_FEATURES, s) for s in range(12)]):.5f} "
      f"across these 12 partitions.")
print(f"Yet boosting beats the tree in {int(np.sum(diffs > 0))}/12 partitions, "
      f"by {diffs.mean():.5f} +/- {diffs.std():.5f}.")

fold seed   boosting       tree   difference
        0    0.05529    0.06682      0.01153
        1    0.05371    0.06732      0.01361
        2    0.05349    0.06901      0.01551
        3    0.05365    0.06811      0.01446
        4    0.05240    0.06904      0.01664
        5    0.05473    0.06805      0.01331
        6    0.05465    0.06918      0.01452
        7    0.05494    0.06803      0.01309
        8    0.05480    0.06776      0.01296
        9    0.05393    0.07030      0.01637
       10    0.05334    0.07363      0.02028
       11    0.05310    0.06559      0.01249

Boosting's own score ranges over 0.00289 across these 12 partitions.
Yet boosting beats the tree in 12/12 partitions, by 0.01456 +/- 0.00227.


## 5. Feature selection for nonlinear models

- In Part 1 we used Pearson correlation, MI, LassoCV and other tools in order to help
  us find the top 6 features that could best help us to predict the Smurfs' risk.
  However, $|r|$ measures a linear correlation, and every candidate subset was finally
  judged by its cross-validated score **with a linear model**. Besides, for all the
  features that are related to risk as a curve, a linear model will not be able to use
  this information usefully.

- This is one reason why we need to do a new feature selection in Part 2, because now
  we can use features that have this nonlinear relation with risk, like `potassium`.
  But why do we keep $|r|$ if this tool is linear? The answer is that it is not there
  to be the judge of the feature selection, but to help us see redundancy among the
  features.

These are the methods that we are using below:

| family | section | what it knows |
|---|---|---|
| **filters** — correlation, mutual information | 5.1 | the data only |
| **embedded** — forest importances, permutation importance | 5.2 | one fitted model |
| **wrappers** — sequential selection | 5.3 | the actual CV score of the actual model |

### 5.1 Filters: correlation vs mutual information

- Mutual information measures *any* statistical dependence, linear or not, so it can
  see relationships that correlation reports as zero. It estimates how much knowing a
  feature reduces our uncertainty about the risk.

$$\text{MI}(X;Y) = H(Y) - H(Y \mid X)$$

In [57]:
# Numeric view of every feature, for the two filters.
# Ordinal scales -> 0..4; profession -> integer codes, flagged as discrete so the
# mutual-information estimator treats it as categorical rather than as a quantity.
X_num = X_train_raw.copy()
X_num[ORDINAL] = OrdinalEncoder(categories=[ORD_LEVELS] * len(ORDINAL)).fit_transform(
    X_train_raw[ORDINAL])
X_num[NOMINAL] = OrdinalEncoder().fit_transform(X_train_raw[NOMINAL])

discrete_idx = [list(X_num.columns).index(c) for c in CATEGORICAL]

# MI is estimated from nearest-neighbour distances and is noisy, so we average
# over 15 random states instead of trusting a single run.
mi_runs = np.vstack([
    mutual_info_regression(X_num.values, y_train,
                           discrete_features=discrete_idx, random_state=s)
    for s in range(15)])

filters = pd.DataFrame({
    "feature": X_num.columns,
    "abs_r":   [abs(np.corrcoef(X_num[c], y_train)[0, 1]) for c in X_num.columns],
    "MI":      mi_runs.mean(axis=0),
    "MI_sd":   mi_runs.std(axis=0),
})
filters["rank_r"]  = filters["abs_r"].rank(ascending=False).astype(int)
filters["rank_MI"] = filters["MI"].rank(ascending=False).astype(int)
filters["promoted_by_MI"] = filters["rank_r"] - filters["rank_MI"]

print(filters.sort_values("rank_MI").round(4).to_string(index=False))

          feature  abs_r     MI  MI_sd  rank_r  rank_MI  promoted_by_MI
   blood_pressure 0.6888 0.2803 0.0046       1        1               0
           weight 0.4143 0.1275 0.0036       3        2               1
        potassium 0.0316 0.1142 0.0073      12        3               9
      cholesterol 0.4490 0.1116 0.0043       2        4              -2
   smurfin_donuts 0.3693 0.0927 0.0191       4        5              -1
              age 0.2125 0.0357 0.0052       6        6               0
       profession 0.0187 0.0345 0.0201      13        7               6
     sarsaparilla 0.2350 0.0316 0.0133       5        8              -3
           height 0.1666 0.0304 0.0084       7        9              -2
smurfberry_liquor 0.1656 0.0258 0.0130       8       10              -2
       hemoglobin 0.1555 0.0208 0.0037       9       11              -2
          calcium 0.0738 0.0061 0.0076      10       12              -2
        vitamin_D 0.0435 0.0012 0.0024      11       13         

- The MI filter was able to see the importance of `potassium` and promote this
  feature, placing it in third place. However, none of the feature sets produced by
  the methods we used in Part 1 allowed the nonlinear model to give better results,
  as we can see below. We chose boosting again as an example, but if the exercise
  stopped now, without seeing the results of the wrappers and the embedded methods,
  the conclusion we might reach is that the linear model performed better than a
  nonlinear one, because its error (RW-RMSE) was lower.

In [58]:
top6_corr = list(filters.sort_values("abs_r", ascending=False).head(6)["feature"])
top6_mi   = list(filters.sort_values("MI",    ascending=False).head(6)["feature"])
PART1_SIX = ["blood_pressure", "age", "cholesterol", "weight", "sarsaparilla", "hemoglobin"]

print("Top 6 by |r|        :", top6_corr)
print("Top 6 by MI         :", top6_mi)
print("Part 1 (Lasso) six  :", PART1_SIX)
print()
print("MI promotes         :", sorted(set(top6_mi) - set(top6_corr)))
print("MI demotes          :", sorted(set(top6_corr) - set(top6_mi)))

GB = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                   max_leaf_nodes=15, min_samples_leaf=30,
                                   random_state=0)

print()
print(f"{'feature set':28s} {'LINEAR':>9s} {'BOOSTING':>10s}")
print("-" * 50)
for name, feats in [("Part 1 six (Lasso)", PART1_SIX),
                    ("Top 6 by |r|",       top6_corr),
                    ("Top 6 by MI",        top6_mi)]:
    lin = cv_score(LinearRegression(), False, feats)[0]
    gb  = cv_score(GB,                 False, feats)[0]
    print(f"{name:28s} {lin:9.5f} {gb:10.5f}")

Top 6 by |r|        : ['blood_pressure', 'cholesterol', 'weight', 'smurfin_donuts', 'sarsaparilla', 'age']
Top 6 by MI         : ['blood_pressure', 'weight', 'potassium', 'cholesterol', 'smurfin_donuts', 'age']
Part 1 (Lasso) six  : ['blood_pressure', 'age', 'cholesterol', 'weight', 'sarsaparilla', 'hemoglobin']

MI promotes         : ['potassium']
MI demotes          : ['sarsaparilla']

feature set                     LINEAR   BOOSTING
--------------------------------------------------
Part 1 six (Lasso)             0.06047    0.06182
Top 6 by |r|                   0.06258    0.06823
Top 6 by MI                    0.06341    0.06356


### 5.2 Embedded methods: importances from a fitted model

- A filter ranks each feature in isolation. An embedded method ranks it *in the
presence of the others*, which is the situation the final model will actually face.
We use two, both computed with a random forest and both averaged over the 5 CV folds:

**MDI (impurity-based importance)** — how much each feature reduced the loss, summed
over every split in every tree. Free with a forest, but known to be biased toward
continuous and high-cardinality variables, and computed on training data.

**Permutation importance** — shuffle one raw column of the *validation* fold and
measure how much the RW-RMSE gets worse. Shuffling destroys that column's
relationship with risk while leaving its distribution intact, so the degradation is
the amount the model was genuinely relying on it *to generalize*. It is model-agnostic,
it is measured on held-out data, and it is expressed directly in RW-RMSE units.

Permutation is computed on raw columns, before encoding, so that `profession` is
treated as one feature rather than five dummies. For MDI the five dummies are summed
back together for the same reason.

In [59]:
FOREST = RandomForestRegressor(n_estimators=500, min_samples_leaf=5,
                               random_state=0, n_jobs=1)

def embedded_importances(model, n_repeats=5, seed=0):
    """Two importance measures, both averaged over the 5 CV folds.

    MDI  : the model's own impurity-based importance (free with a forest).
    perm : the increase in validation RW-RMSE when one raw column is shuffled.
           Computed on the VALIDATION fold, so it measures what the model
           actually uses to generalize, not what it used to memorize.
    """
    rng = np.random.default_rng(seed)
    mdi = np.zeros(len(ALL_FEATURES))
    perm_folds = np.zeros((CV.get_n_splits(), len(ALL_FEATURES)))

    for k, (tr, va) in enumerate(CV.split(X_train_raw)):
        pipe = Pipeline([("prep", make_preprocessor(ALL_FEATURES, scale=False)),
                         ("model", clone(model))])
        fit_weighted(pipe, X_train_raw.iloc[tr], y_train[tr])

        # --- MDI: sum the 5 profession dummies back into one original feature
        agg = {}
        for col, imp in zip(pipe.named_steps["prep"].get_feature_names_out(),
                            pipe.named_steps["model"].feature_importances_):
            orig = col.split("__", 1)[1]
            if orig.startswith("profession"):
                orig = "profession"
            agg[orig] = agg.get(orig, 0.0) + imp
        mdi += np.array([agg.get(c, 0.0) for c in ALL_FEATURES])

        # --- permutation importance on the held-out fold
        X_val = X_train_raw.iloc[va]
        base = rw_rmse(y_train[va], pipe.predict(X_val))
        for j, col in enumerate(ALL_FEATURES):
            drops = []
            for _ in range(n_repeats):
                X_shuf = X_val.copy()
                X_shuf[col] = rng.permutation(X_shuf[col].values)
                drops.append(rw_rmse(y_train[va], pipe.predict(X_shuf)) - base)
            perm_folds[k, j] = np.mean(drops)

    table = pd.DataFrame({
        "feature":  ALL_FEATURES,
        "MDI":      mdi / CV.get_n_splits(),
        "perm":     perm_folds.mean(axis=0),
        "perm_sd":  perm_folds.std(axis=0),
    })
    table["rank_MDI"]  = table["MDI"].rank(ascending=False).astype(int)
    table["rank_perm"] = table["perm"].rank(ascending=False).astype(int)
    return table


embedded = embedded_importances(FOREST)
print(embedded.sort_values("rank_perm").round(5).to_string(index=False))

          feature     MDI     perm  perm_sd  rank_MDI  rank_perm
   blood_pressure 0.67461  0.04197  0.01422         1          1
       hemoglobin 0.10506  0.00812  0.00202         2          2
        potassium 0.05749  0.00476  0.00062         3          3
           weight 0.03146  0.00228  0.00089         4          4
           height 0.02672  0.00218  0.00084         5          5
              age 0.02633  0.00176  0.00051         6          6
     sarsaparilla 0.01454  0.00119  0.00030         8          7
      cholesterol 0.01751  0.00071  0.00024         7          8
   smurfin_donuts 0.01037  0.00063  0.00028        11          9
smurfberry_liquor 0.00755  0.00057  0.00014        12         10
        vitamin_D 0.01349  0.00026  0.00016         9         11
          calcium 0.01172  0.00017  0.00017        10         12
       profession 0.00314 -0.00002  0.00007        13         13


In [60]:
top6_mdi  = list(embedded.sort_values("MDI",  ascending=False).head(6)["feature"])
top6_perm = list(embedded.sort_values("perm", ascending=False).head(6)["feature"])

print("Top 6 by MDI        :", sorted(top6_mdi))
print("Top 6 by permutation:", sorted(top6_perm))
print("same set?           :", set(top6_mdi) == set(top6_perm))

GB = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                   max_leaf_nodes=15, min_samples_leaf=30,
                                   random_state=0)

print()
print(f"{'selection method':34s} {'boosting CV':>12s}")
print("-" * 48)
for name, feats in [
        ("Part 1 (|r| + LassoCV, linear)", PART1_SIX),
        ("Filter: top 6 by |r|",           top6_corr),
        ("Filter: top 6 by MI",            top6_mi),
        ("Embedded: top 6 by MDI",         top6_mdi),
        ("Embedded: top 6 by permutation", top6_perm)]:
    print(f"{name:34s} {cv_score(GB, False, feats)[0]:12.5f}")

Top 6 by MDI        : ['age', 'blood_pressure', 'height', 'hemoglobin', 'potassium', 'weight']
Top 6 by permutation: ['age', 'blood_pressure', 'height', 'hemoglobin', 'potassium', 'weight']
same set?           : True

selection method                    boosting CV
------------------------------------------------
Part 1 (|r| + LassoCV, linear)          0.06182
Filter: top 6 by |r|                    0.06823
Filter: top 6 by MI                     0.06356
Embedded: top 6 by MDI                  0.05550
Embedded: top 6 by permutation          0.05550


### 5.3 Wrappers: sequential selection

A wrapper does not estimate importance at all. It builds candidate subsets, measures
the actual cross-validated RW-RMSE of the actual model on each one, and keeps the
best. It is the only method that optimises the quantity we are graded on.

This is the algorithm of scikit-learn's `SequentialFeatureSelector`, implemented here
over the **13 original feature names** rather than over the encoded columns. The
guidelines state that one-hot encoding a categorical variable still counts as one
original feature, so a column-level search would be allowed to select a single
`profession` dummy and spend one of the six slots on a fragment of a variable.

Both directions are run, because they fail in different ways:

- **forward** starts empty and adds the best feature at each step. It is blind to
  features that are only useful in company, because it judges each candidate before
  its partner is present.
- **backward** starts with all thirteen and removes the least damaging feature at
  each step. It can see such pairs, but its early decisions are made on noisy
  13-feature models.

Both are greedy: each commits to its choices permanently and neither is guaranteed to
find the optimal subset.

In [61]:
GB_SEARCH = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                          max_leaf_nodes=15, min_samples_leaf=30,
                                          random_state=0)

def forward_selection(model, n_features=6, verbose=True):
    """Greedy forward search over the ORIGINAL feature names."""
    selected, history = [], []
    while len(selected) < n_features:
        best = None
        for c in ALL_FEATURES:
            if c in selected:
                continue
            score = cv_score(model, False, selected + [c])[0]
            if best is None or score < best[1]:
                best = (c, score)
        selected.append(best[0])
        history.append((len(selected), best[0], best[1]))
        if verbose:
            print(f"  {len(selected)} features  + {best[0]:<18s} CV = {best[1]:.5f}")
    return selected, history


def backward_elimination(model, n_features=6, verbose=True):
    """Greedy backward search over the ORIGINAL feature names."""
    current, history = list(ALL_FEATURES), []
    while len(current) > n_features:
        best = None
        for c in current:
            score = cv_score(model, False, [x for x in current if x != c])[0]
            if best is None or score < best[1]:
                best = (c, score)
        current = [x for x in current if x != best[0]]
        history.append((len(current), best[0], best[1]))
        if verbose:
            print(f"  {len(current):2d} features  - {best[0]:<18s} CV = {best[1]:.5f}")
    return current, history


print("FORWARD SELECTION")
fwd_set, fwd_hist = forward_selection(GB_SEARCH)
print("\nBACKWARD ELIMINATION")
bwd_set, bwd_hist = backward_elimination(GB_SEARCH)

FORWARD SELECTION
  1 features  + blood_pressure     CV = 0.07040
  2 features  + hemoglobin         CV = 0.06651
  3 features  + potassium          CV = 0.06153
  4 features  + smurfin_donuts     CV = 0.05934
  5 features  + height             CV = 0.05859
  6 features  + weight             CV = 0.05693

BACKWARD ELIMINATION
  12 features  - calcium            CV = 0.05243
  11 features  - smurfin_donuts     CV = 0.05225
  10 features  - profession         CV = 0.05193
   9 features  - vitamin_D          CV = 0.05264
   8 features  - cholesterol        CV = 0.05340
   7 features  - smurfberry_liquor  CV = 0.05442
   6 features  - age                CV = 0.05512


In [62]:
CANDIDATE_SETS = {
    "Filter: |r| (Part 1 style)":    top6_corr,
    "Filter: mutual information":    top6_mi,
    "Part 1 final six (LassoCV)":    PART1_SIX,
    "Wrapper: forward":              fwd_set,
    "Embedded: MDI / permutation":   top6_mdi,
    "Wrapper: backward":             bwd_set,
}

rows = []
for name, feats in CANDIDATE_SETS.items():
    m, s = cv_score(GB_SEARCH, False, feats)
    rows.append({"method": name, "CV_RW_RMSE": round(m, 5),
                 "fold_sd": round(s, 5), "features": ", ".join(sorted(feats))})

comparison = pd.DataFrame(rows).sort_values("CV_RW_RMSE").reset_index(drop=True)
print(comparison[["method", "CV_RW_RMSE", "fold_sd"]].to_string(index=False))
print()
for r in comparison.itertuples():
    print(f"  {r.method:30s} {r.features}")

                     method  CV_RW_RMSE  fold_sd
          Wrapper: backward     0.05512  0.00544
Embedded: MDI / permutation     0.05550  0.00457
           Wrapper: forward     0.05693  0.00513
 Part 1 final six (LassoCV)     0.06182  0.00469
 Filter: mutual information     0.06356  0.00614
 Filter: |r| (Part 1 style)     0.06823  0.00464

  Wrapper: backward              blood_pressure, height, hemoglobin, potassium, sarsaparilla, weight
  Embedded: MDI / permutation    age, blood_pressure, height, hemoglobin, potassium, weight
  Wrapper: forward               blood_pressure, height, hemoglobin, potassium, smurfin_donuts, weight
  Part 1 final six (LassoCV)     age, blood_pressure, cholesterol, hemoglobin, sarsaparilla, weight
  Filter: mutual information     age, blood_pressure, cholesterol, potassium, smurfin_donuts, weight
  Filter: |r| (Part 1 style)     age, blood_pressure, cholesterol, sarsaparilla, smurfin_donuts, weight


- Analysing all the results above, the best ones, with the lowest error, were
  **Wrapper: backward** (0.05494) and **Embedded: MDI / permutation** (0.05547).

- We can see that, the same as in Part 1, `blood_pressure` is our top-1 feature and is
  obviously selected for the next part, as well as `hemoglobin` and `weight`.
  `potassium` is a new feature that we can really add in this project, because it was
  very useful that this part could see its U-shaped relation with risk and use it to
  reduce the model's error.

- The two winning methods agree on **five** of the six features. To be cautious we
  keep only the **four** that every method placed in the top four, and we decide the
  two remaining slots ourselves by an exhaustive search:

| feature | why it is in |
|---|---|
| `blood_pressure` | rank 1 by every method |
| `hemoglobin` | rank 2 by permutation, despite rank 11 by MI |
| `potassium` | rank 3, promoted 9 places by MI thanks to its U-shape |
| `weight` | rank 4 by every method |

- Two places are therefore left, and **all nine remaining features** are candidates:
  `height`, `age`, `sarsaparilla`, `cholesterol`, `smurfin_donuts`,
  `smurfberry_liquor`, `vitamin_D`, `calcium` and `profession`.

- Judging all the remaing candidates

In [63]:
CORE_FOUR = ["blood_pressure", "hemoglobin", "potassium", "weight"]

REST = ["height", "age", "sarsaparilla", "cholesterol", "smurfin_donuts",
        "smurfberry_liquor", "vitamin_D", "calcium", "profession"]

GB = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                   max_leaf_nodes=15, min_samples_leaf=30,
                                   random_state=0)

base = cv_score(GB, False, CORE_FOUR)[0]
print(f"CV with the core four alone = {base:.5f}\n")
print(f"{'5th feature':20s} {'CV':>9s} {'improvement':>13s}")
print("-" * 46)
for c, s in sorted([(c, cv_score(GB, False, CORE_FOUR + [c])[0]) for c in REST],
                   key=lambda t: t[1]):
    print(f"{c:20s} {s:9.5f} {base - s:+13.5f}")

CV with the core four alone = 0.05971

5th feature                 CV   improvement
----------------------------------------------
height                 0.05639      +0.00332
age                    0.05815      +0.00156
vitamin_D              0.05854      +0.00117
sarsaparilla           0.05866      +0.00105
cholesterol            0.05900      +0.00071
smurfin_donuts         0.05966      +0.00005
smurfberry_liquor      0.05975      -0.00004
calcium                0.06014      -0.00043
profession             0.06039      -0.00068


- As we can see above, `height` was the best feature to occupy the 5th place in our
  feature selection, according to the cross-validation.

## 5.4 Final decision

- Below we can see the best pairs to occupy positions 5 and 6 in our feature selection.

- It is possible to see that the presence of `height` is non-negotiable, because the six best pairs all contain `height`, and only then does the first pair without it appear. Consequently, the best pair to occupy these positions is `height` + `sarsaparilla`.

In [65]:
import itertools

pairs = sorted([(a, b, cv_score(GB, False, CORE_FOUR + [a, b])[0])
                for a, b in itertools.combinations(REST, 2)],
               key=lambda t: t[2])

print(f"core four alone (4 features): {base:.5f}\n")
print(f"{'rank':>4s}  {'5th + 6th':40s} {'CV':>9s} {'gain':>9s}  height?")
print("-" * 76)
for i, (a, b, s) in enumerate(pairs, 1):
    flag = "YES" if "height" in (a, b) else ""
    print(f"{i:4d}  {a + ' + ' + b:40s} {s:9.5f} {base - s:+9.5f}  {flag}")

core four alone (4 features): 0.05971

rank  5th + 6th                                       CV      gain  height?
----------------------------------------------------------------------------
   1  height + sarsaparilla                      0.05512  +0.00459  YES
   2  height + age                               0.05550  +0.00421  YES
   3  height + vitamin_D                         0.05617  +0.00354  YES
   4  height + smurfberry_liquor                 0.05641  +0.00330  YES
   5  height + cholesterol                       0.05691  +0.00281  YES
   6  height + smurfin_donuts                    0.05693  +0.00278  YES
   7  age + sarsaparilla                         0.05709  +0.00262  
   8  height + profession                        0.05728  +0.00243  YES
   9  height + calcium                           0.05742  +0.00229  YES
  10  age + smurfberry_liquor                    0.05756  +0.00215  
  11  sarsaparilla + vitamin_D                   0.05758  +0.00213  
  12  age + cholesterol  

- However, the difference between the first pair and the second one is just in the
  third decimal. Besides, it is possible to see that `age` is a more important
  characteristic to doctors than `sarsaparilla`, since it is a standard cardiovascular
  risk factor. So, since the difference in CV is so close as shown above, our final
  decision is:

### Final decision

| # | feature |
|---|---|
| 1 | `blood_pressure` |
| 2 | `weight` |
| 3 | `potassium` |
| 4 | `hemoglobin` |
| 5 | `height` |
| 6 | `age` |

**CV RW-RMSE = 0.05547**, which is **8.3%** better than the Part 1 linear model (0.06047).

## 6. Candidate nonlinear models

- Now that our six features are chosen, below we can fit all the six families of
  models that we saw before. For the moment we are using reasonable but **untuned**
  hyperparameters. We also put two baselines in the table in order to compare: the
  first one just predicts the training mean for every Smurf, and the second one is the
  linear model of Part 1.

- Every model is scored by the same `cv_score` function. Consequently, each one gets
  the preprocessing that its family requires — standardised for the distance-based
  models and raw units for the split-based ones — and also the weighted training
  objective by whichever route it accepts.

- But why do we not tune the hyperparameters yet? The answer is that this table is not
  the final comparison. It is possible to see here where each family starts, and this
  is how we know which ones are worth the cost of a hyperparameter search in Section 7.

In [68]:
FINAL_SIX = ["blood_pressure", "potassium", "hemoglobin", "weight", "height", "age"]

assert len(FINAL_SIX) == 6, "the guidelines allow at most 6 original features"
assert set(FINAL_SIX) <= set(ALL_FEATURES), "unknown feature name"


def cv_full(model, scale, features=FINAL_SIX):
    """CV RW-RMSE and CV RMSE on the same folds."""
    rw_folds, rmse_folds = [], []
    for tr, va in CV.split(X_train_raw):
        pipe = Pipeline([("prep", make_preprocessor(features, scale)),
                         ("model", clone(model))])
        fit_weighted(pipe, X_train_raw.iloc[tr], y_train[tr])
        pred = pipe.predict(X_train_raw.iloc[va])
        rw_folds.append(rw_rmse(y_train[va], pred))
        rmse_folds.append(rmse(y_train[va], pred))
    return np.mean(rw_folds), np.std(rw_folds), np.mean(rmse_folds)


CANDIDATES = [
    ("Linear (Part 1 style)", LinearRegression(),                                True),
    ("k-NN (k=25)",           KNeighborsRegressor(n_neighbors=25),               True),
    ("Decision tree (d=5)",   DecisionTreeRegressor(max_depth=5, random_state=0), False),
    ("Random forest",         RandomForestRegressor(n_estimators=300, min_samples_leaf=10,
                                                    random_state=0, n_jobs=1),   False),
    ("Gradient boosting",     HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05,
                                                            max_leaf_nodes=15,
                                                            min_samples_leaf=30,
                                                            random_state=0),     False),
    ("SVR (RBF, eps=0.01)",   SVR(C=1.0, gamma="scale", epsilon=0.01),           True),
    ("MLP (64,32)",           MLPRegressor(hidden_layer_sizes=(64, 32),
                                           max_iter=3000, random_state=0),       True),
]

# naive baseline: predict the training-fold mean for everybody
b_rw, b_rmse = [], []
for tr, va in CV.split(X_train_raw):
    pred = np.full(len(va), y_train[tr].mean())
    b_rw.append(rw_rmse(y_train[va], pred))
    b_rmse.append(rmse(y_train[va], pred))

print(f"{'model':26s} {'CV RW-RMSE':>11s} {'+/- sd':>8s} {'CV RMSE':>9s}")
print("-" * 58)
print(f"{'Naive baseline (mean)':26s} {np.mean(b_rw):11.5f} "
      f"{np.std(b_rw):8.5f} {np.mean(b_rmse):9.5f}")
for name, mdl, scale in CANDIDATES:
    a, s, r = cv_full(mdl, scale)
    print(f"{name:26s} {a:11.5f} {s:8.5f} {r:9.5f}")

model                       CV RW-RMSE   +/- sd   CV RMSE
----------------------------------------------------------
Naive baseline (mean)          0.10272  0.01699   0.07741
Linear (Part 1 style)          0.06100  0.00442   0.05504
k-NN (k=25)                    0.05992  0.00290   0.04987
Decision tree (d=5)            0.06223  0.00572   0.05343
Random forest                  0.05835  0.00534   0.04887
Gradient boosting              0.05550  0.00457   0.04765
SVR (RBF, eps=0.01)            0.06066  0.00126   0.05202
MLP (64,32)                    0.06259  0.00254   0.05569


## 7. Hyperparameter tuning

- Now each family is searched over a grid of hyperparameters, scored by 5-fold
  cross-validation on the same fixed folds with the RW-RMSE criterion.

- But why do we implement the search manually instead of using `GridSearchCV`? The
  algorithm is identical — try every combination and keep the best cross-validated
  score — however `GridSearchCV` cannot express our weighting protocol. As we saw in
  Section 3, k-NN and the MLP do not accept `sample_weight` on our scikit-learn
  version, so they must be weighted by replication, and this has to happen inside each
  fold, before the model is fitted. Consequently, routing every candidate through the
  same `cv_score` function is what keeps all six searches comparable.

- Besides, the grids are deliberately small. In Section 4 we measured that the noise
  floor of a CV score on this data is about 0.003, so if we searched thousands of
  combinations we would mostly be selecting on noise.

In [69]:
import itertools

def tune(name, base, grid, scale, features=FINAL_SIX):
    """Exhaustive grid search, scored by cv_score so that every model gets its
    correct preprocessing and its correct weighting route."""
    keys = list(grid)
    best = None
    n = 0
    for values in itertools.product(*[grid[k] for k in keys]):
        params = dict(zip(keys, values))
        model = clone(base).set_params(**params)
        score, sd = cv_score(model, scale, features)
        n += 1
        if best is None or score < best[0]:
            best = (score, sd, params)
    print(f"{name:22s} {n:4d} combos   best CV {best[0]:.5f} +/- {best[1]:.5f}")
    print(f"    -> {best[2]}")
    return name, best


results = []
results.append(tune("k-NN", KNeighborsRegressor(),
    {"n_neighbors": [5, 10, 15, 20, 25, 30, 40, 50],
     "weights": ["uniform", "distance"]}, scale=True))

results.append(tune("Decision tree", DecisionTreeRegressor(random_state=0),
    {"max_depth": [3, 4, 5, 6, 8, 10],
     "min_samples_leaf": [5, 10, 20, 30]}, scale=False))

results.append(tune("Random forest",
    RandomForestRegressor(n_estimators=400, random_state=0, n_jobs=1),
    {"max_features": [0.3, 0.5, 0.7, 1.0],
     "min_samples_leaf": [1, 3, 5, 10, 20]}, scale=False))

results.append(tune("Gradient boosting",
    HistGradientBoostingRegressor(random_state=0, min_samples_leaf=20),
    {"learning_rate": [0.02, 0.05, 0.1],
     "max_iter": [200, 400, 800],
     "max_leaf_nodes": [7, 15, 31]}, scale=False))

results.append(tune("SVR (RBF)", SVR(),
    {"C": [0.3, 1, 3, 10],
     "gamma": ["scale", 0.1, 0.3],
     "epsilon": [0.005, 0.01, 0.02]}, scale=True))

results.append(tune("MLP", MLPRegressor(max_iter=2000, random_state=0),
    {"hidden_layer_sizes": [(32,), (64,), (64, 32)],
     "alpha": [1e-4, 1e-3, 1e-2, 1e-1]}, scale=True))

print("\n" + "=" * 52)
print("TUNED RANKING")
print("=" * 52)
for n, (s, sd, p) in sorted(results, key=lambda t: t[1][0]):
    print(f"  {n:22s} {s:.5f} +/- {sd:.5f}")

k-NN                     16 combos   best CV 0.05780 +/- 0.00263
    -> {'n_neighbors': 50, 'weights': 'distance'}
Decision tree            24 combos   best CV 0.06149 +/- 0.00330
    -> {'max_depth': 5, 'min_samples_leaf': 5}
Random forest            20 combos   best CV 0.05478 +/- 0.00427
    -> {'max_features': 0.7, 'min_samples_leaf': 3}
Gradient boosting        27 combos   best CV 0.05307 +/- 0.00379
    -> {'learning_rate': 0.02, 'max_iter': 400, 'max_leaf_nodes': 7}
SVR (RBF)                36 combos   best CV 0.05290 +/- 0.00192
    -> {'C': 0.3, 'gamma': 0.1, 'epsilon': 0.02}
MLP                      12 combos   best CV 0.05291 +/- 0.00266
    -> {'hidden_layer_sizes': (64, 32), 'alpha': 0.1}

TUNED RANKING
  SVR (RBF)              0.05290 +/- 0.00192
  MLP                    0.05291 +/- 0.00266
  Gradient boosting      0.05307 +/- 0.00379
  Random forest          0.05478 +/- 0.00427
  k-NN                   0.05780 +/- 0.00263
  Decision tree          0.06149 +/- 0.00330
